# HALE: an epidemic that reacts to what an LLM says people will do

[Moon et al. (2026)](https://arxiv.org/abs/2607.06757) couple an individual-based SIR model
to a language model. The ABM spreads COVID-19 over an activity-based contact network of
Salt Lake County; every week, the LLM is asked whether a given kind of person still goes
out given the reported prevalence, and the network is rewired accordingly. Their headline:
the ABM alone overestimates the epidemic, and the feedback loop is what fixes it.

The framework needs nothing new to run this. `llm:choose` is a SPARQL UDF, so the LLM is a
function inside a rule; the rest is four rules in
[`skabm/hale.py`](../skabm/hale.py).

| HALE | here |
|---|---|
| 1.13M agents, property vector *P* | a `Person` population |
| activity network *G(t)*, 22 activities | an `Edge` population — `ottr.edge_template` already carries `src`/`dst`/`weight`, and `kind` widens it |
| deliberate vs non-deliberate actions | `def:kind` = `"out"` / `"home"`; only `"out"` edges are switched off |
| 1,552 LLM agents, one per demographic group | a `Group` population; `\|N\| >> \|L\|` is the `def:group` join |
| Δt_LLM = k·Δt_ABM, weekly | prevalence **bucketed** in the prompt, so the memo collapses the calls |
| structured yes/no via Outlines | `choices={"yes": 1.0, "no": 0.0}`, mapped inside the UDF |
| Llama-3.1-8B on vLLM, Frontier | one hosted model, or none — three of the four arms below need no account |

Scaled down honestly: the population is synthetic rather than UrbanPop, there is one static
weighted network rather than 3,696 hourly ones, and the run is 60 days rather than 22 weeks.
Nothing in that list is a limitation of the framework; each is a dataset we do not have.

In [16]:
import getpass
import os

# Interactive setup for the LLM arm.  Wrapped because `getpass` raises headlessly
# (nbmake, CI, any non-tty frontend); there the notebook falls through to its
# no-key path and the three offline arms still run.
try:
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
    if "ANTHROPIC_WORKSPACE_ID" not in os.environ:
        # Only identity-linked keys need this; leave blank for a classic key.
        workspace = getpass.getpass("Anthropic workspace id (blank if unsure): ").strip()
        if workspace:
            os.environ["ANTHROPIC_WORKSPACE_ID"] = workspace
except Exception:
    print("no interactive input here — running the offline arms only")

In [10]:

from time import time

import matplotlib.pyplot as plt
import polars as pl
import polars_random as pr
from maplib import xsd

from skabm.hale import GOES_OUT, HALE_PARAMS, HALE_UPDATE_RULES
from skabm.rules import LLM_NS, register_llm, register_polars_random
from skabm.simulation import RDFSimulator

N_PEOPLE, N_DAYS, SEED = 800, 60, 0
HOUSEHOLD, OUT_DEGREE = 4, 17     # tuned so mean degree lands near the paper's 22.41
MODEL = "claude-haiku-4-5-20251001"

# The paper's grouping is municipality x sex x race x age band -> 1,552 groups.
# We vary municipality and age band and hold the other two fixed: the age
# gradient is the paper's most interesting LLM result (Figure 4) and the
# municipality spread its second (Figure 5), while every extra dimension
# multiplies the number of distinct prompts.
MUNICIPALITIES = ["Salt Lake City", "Bluffdale", "Alta"]
AGE_BANDS = ["0-18", "19-24", "25-34", "35-44", "45-59", "over 60"]

groups = pl.DataFrame(
    [
        {
            "id": f"g_{m.replace(' ', '')}_{a.replace(' ', '')}",
            "municipality": m,
            "age_band": a,
            "sex": "female",
            "race": "Asian",
        }
        for m in MUNICIPALITIES
        for a in AGE_BANDS
    ]
)
groups.head(3)

id,municipality,age_band,sex,race
str,str,str,str,str
"""g_SaltLakeCity_0-18""","""Salt Lake City""","""0-18""","""female""","""Asian"""
"""g_SaltLakeCity_19-24""","""Salt Lake City""","""19-24""","""female""","""Asian"""
"""g_SaltLakeCity_25-34""","""Salt Lake City""","""25-34""","""female""","""Asian"""


## The population, and the two kinds of contact

A person belongs to a household and to a demographic group. The household is the
non-deliberate contact — you do not decide to live with your family — and everything else is
deliberate. That distinction is the whole behavioural mechanism: when a group declines to go
out, its `"out"` edges stop conducting and its `"home"` edges do not.

Edges are stored in both directions because the transmission rule reads them directionally
(susceptible as `src`), which is cheaper than a `UNION` in the hot rule.

In [11]:
people = (
    pl.DataFrame({"n": range(N_PEOPLE)})
    .with_columns(
        id=pl.format("p{}", pl.col("n")),
        group=pl.Series(groups["id"].sample(N_PEOPLE, with_replacement=True, seed=SEED)),
        household=pl.col("n") // HOUSEHOLD,
        infected=(pl.col("n") < 5).cast(pl.Float64),   # five index cases
        recovered=0.0,
    )
    .drop("n")
)

home = (
    people.select("id", "household")
    .join(people.select(pl.col("id").alias("other"), "household"), on="household")
    .filter(pl.col("id") != pl.col("other"))
    .select(src="id", dst="other", kind=pl.lit("home"), weight=pl.lit(1.0))
)
n_out = N_PEOPLE * OUT_DEGREE // 2
out = (
    pl.DataFrame(
        {
            "src": people["id"].sample(n_out, with_replacement=True, seed=SEED + 1),
            "dst": people["id"].sample(n_out, with_replacement=True, seed=SEED + 2),
        }
    )
    .filter(pl.col("src") != pl.col("dst"))
    .with_columns(kind=pl.lit("out"), weight=pl.lit(1.0))
)
edges = (
    pl.concat([home, out])
    .unique(subset=["src", "dst"])
    .pipe(lambda d: pl.concat([d, d.rename({"src": "dst", "dst": "src"}).select(d.columns)]))
    .with_row_index("i")
    .with_columns(id=pl.format("e{}", pl.col("i")))
    .drop("i")
)
print(f"{people.height} people, {edges.height} directed edges, "
      f"mean degree {edges.height / people.height:.1f} (paper: 22.41)")
edges.head(3)

800 people, 18266 directed edges, mean degree 22.8 (paper: 22.41)


src,dst,kind,weight,id
str,str,str,f64,str
"""p214""","""p628""","""out""",1.0,"""e0"""
"""p183""","""p646""","""out""",1.0,"""e1"""
"""p737""","""p775""","""out""",1.0,"""e2"""


## The rules

Four, in [`skabm/hale.py`](../skabm/hale.py), and the order is the simultaneity convention:

1. **`PREVALENCE`** — one county-wide aggregate, copied onto every group, because the
   paper's prompt states a county figure and not a neighbourhood one. It runs *first*, so
   the number the model sees is the state the decision is actually taken on.
2. **`DECIDE`** — builds the paper's prompt with `CONCAT` and binds `llm:choose` to
   `def:goes_out`.
3. **`RECOVER`** — I → R at rate γ, before transmission, so nobody infected this tick
   recovers in the same tick.
4. **`INFECT`** — one Bernoulli draw per *contact*, not per agent, so a person with more
   infected contacts gets more draws. The behavioural feedback is a single `FILTER`:
   a `"home"` edge always conducts, an `"out"` edge only while both endpoints' groups are
   still going out.

The prompt is the paper's, verbatim:

```
Behavior prediction task
Location: Salt Lake County, UT
Context: 0.02% people infected by COVID-19
Person: Asian female, age 25-34, lives in Bluffdale, Salt Lake County, UT
Question: Considering the infected percentage and a person's behavior based on
          demographic factors, will this person go to a public place?
```

Parameters are theirs too: Omicron R₀ = 9.5 and γ = 1/5 per day, with transmissibility
adjusted by mean degree — β = R₀γ/⟨k⟩ per contact per day.

In [12]:
print(HALE_PARAMS)
print(f"beta = 9.5 * 0.2 / 22.41 = {HALE_PARAMS['beta']:.4f} per contact per day")

{'beta': 0.08478357875948238, 'gamma': 0.2, 'bucket': 1000.0}
beta = 9.5 * 0.2 / 22.41 = 0.0848 per contact per day


## Four deciders, one interface

A decider is whatever registers `llm:choose`. Three of them need no account:

* **free** — everyone always goes out. The epidemic with no behaviour at all.
* **ablation** — the paper's own ABM-only baseline, which "randomly deactivate[s] 35% of
  contacts" to match HALE's average link reduction. An `"out"` edge needs *both* endpoints
  out, so the per-group probability is √0.65 ≈ 0.806, not 0.65 — a detail worth doing
  arithmetic on rather than eyeballing.
* **responsive** — a stub whose decline probability rises with the reported prevalence,
  shaped like the paper's Figure 4. It is not a model of anything; it is there so the
  feedback loop is visible without a key.
* **claude** — the real thing, via `register_llm`.

Note the offline three redraw every tick, so they deliberately bypass the memo; only the
LLM decider memoizes, because only it is expensive.

In [13]:
def register_fixed(p_out: float):
    """Everyone independently goes out with probability `p_out`, redrawn each tick."""

    def register(model):
        def _choose(df: pl.DataFrame) -> pl.Series:
            return (pr.uniform(0.0, 1.0, size=len(df)) < p_out).cast(pl.Float64).alias("out")

        model.add_udf(LLM_NS + "choose", _choose, xsd.double, [xsd.string])

    return register


def register_responsive(scale: float = 20.0):
    """Decline probability rising with the prevalence *read back out of the prompt* —
    the same string the model would see, so the coupling under test is identical."""

    def register(model):
        def _choose(df: pl.DataFrame) -> pl.Series:
            pct = df["0"].str.extract(r"Context: ([0-9.]+)%").cast(pl.Float64)
            decline = (pct * scale / 100.0).clip(0.0, 0.8)
            return (pr.uniform(0.0, 1.0, size=len(df)) > decline).cast(pl.Float64).alias("out")

        model.add_udf(LLM_NS + "choose", _choose, xsd.double, [xsd.string])

    return register


# An identity-linked API key must name the workspace it acts in, or every call
# comes back 400 "anthropic-workspace-id is required".  register_llm forwards
# unknown kwargs to the LangChain constructor, so the fix is a header and not a
# code change; the id is in the Anthropic Console under Settings -> Workspaces.
WORKSPACE = os.environ.get("ANTHROPIC_WORKSPACE_ID")
HEADERS = {"anthropic-workspace-id": WORKSPACE} if WORKSPACE else None


def register_claude(model):
    return register_llm(model, chat=MODEL, choices=GOES_OUT, default_headers=HEADERS)


def run(decider) -> pl.DataFrame:
    sim = RDFSimulator(
        init_rules=(),
        update_rules=HALE_UPDATE_RULES,
        # bucket=1e2 rounds prevalence to whole percent in the prompt.  The paper
        # quotes two decimals; at that resolution every tick invents a new
        # sentence and the memo never hits.  This is the cost knob.
        params=dict(HALE_PARAMS, bucket=1e2),
        n_periods=N_DAYS,
        udfs=(register_polars_random, decider),
        random_seed=SEED,
    )
    frame = pl.DataFrame(sim.fit_iter({"Person": people, "Group": groups, "Edge": edges}))
    cumulative = frame["sig__AVG__Person__infected"] + frame["sig__AVG__Person__recovered"]
    return frame.select(
        "t",
        prevalence="sig__AVG__Person__infected",
        attack_rate=cumulative,
        incidence=cumulative.diff().fill_null(cumulative[0]),
        going_out="sig__AVG__Group__goes_out",
    )

## The three arms that need no account

In [14]:
t0 = time()
arms = {
    "no behaviour": run(register_fixed(1.0)),
    "ABM-only, 35% cut": run(register_fixed(0.65 ** 0.5)),
    "responsive (stub)": run(register_responsive()),
}
print(f"{len(arms)} runs of {N_DAYS} days in {time() - t0:.1f}s")

pl.DataFrame(
    {
        "arm": list(arms),
        "attack rate": [a["attack_rate"][-1] for a in arms.values()],
        "peak day": [int(a["incidence"].arg_max()) + 1 for a in arms.values()],
        "peak incidence": [a["incidence"].max() for a in arms.values()],
        "mean going out": [a["going_out"].mean() for a in arms.values()],
    }
)

3 runs of 60 days in 3.6s


arm,attack rate,peak day,peak incidence,mean going out
str,f64,i64,f64,f64
"""no behaviour""",1.0,7,0.2375,1.0
"""ABM-only, 35% cut""",0.99125,7,0.17,0.800926
"""responsive (stub)""",0.45125,13,0.03625,0.358333


## The arm that asks a model

~18 groups × the handful of prevalence buckets the epidemic actually visits, so the memo
holds this to a few dozen completions for the whole run. Skipped with a printed reason when
there is no key, so everything above still stands.

Setup: `uv sync --extra llm` and `ANTHROPIC_API_KEY`. If your key is **identity-linked**,
also export `ANTHROPIC_WORKSPACE_ID` (Anthropic Console → Settings → Workspaces) — such keys
must name a workspace on every request, and without it every call returns
`400 anthropic-workspace-id is required`. `provider="aopenai"` with a `base_url` points the
same UDF at a local server instead.

In [15]:
if os.environ.get("ANTHROPIC_API_KEY"):
    t0 = time()
    arms["HALE (claude)"] = run(register_claude)
    print(f"HALE arm in {time() - t0:.1f}s")
else:
    print("no ANTHROPIC_API_KEY in this kernel — skipping the HALE arm.")
    print("the three offline arms below are unaffected.")

MaplibException: UDF function error: UDF '<urn:llm:choose>': failed to call UDF: RuntimeError: AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'anthropic-workspace-id is required when authenticating with an identity-linked API key; send the id of the workspace this request acts in.'}, 'request_id': None}

This error occurred in the following expression:
	as_struct("prompt", null.alias("system")).python_udf()


## Figure 3: does a static contact cut stand in for behaviour?

The paper's central comparison. Their ABM-only run deactivates 35% of contacts — the same
*average* link reduction HALE produces — and still misses the peak, because a constant cut
cannot arrive late and cannot deepen as cases rise. The feedback loop is not equivalent to
its own average.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
for (label, frame), color in zip(arms.items(), ["tab:grey", "tab:blue", "tab:green", "tab:orange"]):
    week = ((frame["t"] - 1) // 7).alias("week")
    weekly = frame.with_columns(week).group_by("week").agg(pl.col("incidence").sum()).sort("week")
    top.plot(weekly["week"], weekly["incidence"], marker="o", color=color, label=label)
    bottom.plot(frame["t"], 1 - frame["going_out"], color=color, label=label)

top.set_ylabel("weekly incidence (share of population)")
top.set_title(f"Salt Lake County, scaled to {N_PEOPLE} agents — Omicron, R0 = 9.5")
top.legend()
bottom.set_ylabel("probability of decline")
bottom.set_xlabel("day")
fig.tight_layout()

The lower panel is the paper's Figure 4 in miniature: the share of groups declining to go
out, over time. In the paper it rises gradually and never spikes — "human behavior changes
dynamically in response to surroundings, but the changes occur gradually" — and the age
gradient is the interesting part, with the over-60 and 0–18 groups consistently most likely
to decline.

That gradient is a property of the model, not of the simulation, so it can be read directly:

In [ ]:
def elicited_gradient(prevalence_pct: float = 2.0) -> pl.DataFrame:
    """The paper's Figure 4, asked directly: decline probability by age band."""
    prompts = [
        "Behavior prediction task\n"
        "Location: Salt Lake County, UT\n"
        f"Context: {prevalence_pct}% people infected by COVID-19\n"
        f"Person: Asian female, age {band}, lives in Salt Lake City, Salt Lake County, UT\n"
        "Question: Considering the infected percentage and a person's behavior "
        "based on demographic factors, will this person go to a public place?"
        for band in AGE_BANDS
    ]
    answers = (
        pl.DataFrame({"prompt": prompts})
        .select(
            pl.col("prompt").llm.aanthropic(
                model=MODEL, temperature=0, on_error="raise", default_headers=HEADERS
            )
        )
        .to_series()
    )
    return pl.DataFrame({"age_band": AGE_BANDS, "answer": answers})


elicited_gradient() if "HALE (claude)" in arms else "no key — see the run above"

## What is faithful, and what is not

**Faithful.** The coupling: a group-level decision, written as one triple, read by every
agent through a join, gating exactly the deliberate contacts. The prompt, verbatim. The
epidemiology: individual-based SIR with a per-contact hazard, Omicron's R₀ and γ, β scaled
by mean degree. The comparison the paper rests on, including its 35% ablation. And the
scalability argument — the LLM is asked once per group per situation, which here falls out
of memoizing the prompt rather than being built.

**Not faithful.** One static weighted network instead of 3,696 hourly activity networks
rebuilt from NHTS schedules and H3 cells; a synthetic population instead of UrbanPop matched
to PUMS; 18 groups instead of 1,552; one stochastic replicate instead of 30; no comparison
against observed CDC case data, which is what would make any of this a validation rather
than a demonstration.

**And one thing the paper does that this deliberately does not.** HALE samples at
`temperature=0.2` on purpose: it reads a *probability* of decline off each group, and notes
that at temperature 0 the model declines almost always while at 0.7 every group sits near
0.5. `register_llm` pins `temperature=0` because the prompt memo assumes an answer is a
function of its prompt. To reproduce their Figure 4 properly you want the opposite — sampling
on, memo off, several draws per group — which is a different measurement wearing the same
plumbing.